# Exploration des données dans le dossier bronze/ dans S3

In [43]:
import yaml
import boto3
import pandas as pd
import pyarrow
import io

## To-do list
1. Structure & volume
  - Schéma de chaque fichier (colonnes, types inférés)                                    
  - Nombre de lignes par fichier                                                                    
                                                                                          
  2. Clés & identifiants                                                                
  - Unicité de ID_REFA_LDA — est-ce une vraie clé ?                                       
  - Présence des anciens IDs 2015-2016 vs nouveaux IDs                                    
  - Taux de correspondance avec le référentiel ZdC                                        
                                                                                          
  3. Qualité des données                                                                  
  - Taux de nulls par colonne                                                             
  - Valeurs aberrantes (ex : 0 validations, dates hors périmètre)                         
  - Doublons                                                                            
                                                                                          
  4. Couverture temporelle
  - Toutes les années 2015-2024 sont-elles présentes ?                                    
  - Continuité des données (pas de trou sur une période)                                  
  
  5. Joinabilité                                                                          
  - Les ID_REFA_LDA des validations matchent-ils bien le référentiel accessibilité ?    
  - Stations présentes dans un fichier mais absentes de l'autre   

## **rappel** 

Cette étape de data exploration survient après l'ingestion réalisée dans S3. Les fichiers sources ont été exportés en .parquet dans S3 dans le dossier bronze/. Trois types de fichiers :
- `NB_FER` : séparés dans des dossiers par année  avec deux fichiers chacun (un par semestre)
- `accessibilité` : un seul fichier .parquet à la racine de bronze/
- `references` : un seul fichier .parquet à la racine de bronze/


Il y a logiquement : 

- NB_FER : (2024-2016)x2 = 16 fichiers -> on n'en garde que 3 (de l'année 2024)
- accessibilité = 1 fichier
- références = 1 fichier

--> 18 fichiers

## 0 - On passe nos trois fichiers dans S3 en DataFrame pour exploration : 
 - NB_FER_2024 (données de S1 + T3 + T4 de 2024)
 - ACCESSIBILITE
 - REFERENCE
 ---


### CONFIG CLIENT

In [44]:
s3_client = boto3.client('s3')

In [45]:
file = open("../../config/config.yaml")
yaml_file = yaml.safe_load(file)
file.close()

In [46]:
bucket = yaml_file['S3']['bucket']
prefix = yaml_file['S3']['chemins_dossiers']['bronze']

In [47]:
list = s3_client.list_objects_v2(
Bucket = bucket,
Prefix = prefix
)

In [48]:
list.keys()

dict_keys(['ResponseMetadata', 'IsTruncated', 'Contents', 'Name', 'Prefix', 'MaxKeys', 'EncodingType', 'KeyCount'])

In [49]:
list_objects = list['Contents']

### NB_FER_2024

-> Pour notre analyse, on ne garde que les fichiers de l'année 2024

In [50]:
scope_objects = [i['Key'] for i in list_objects if '2024'in i['Key']]

In [51]:
dfs = []                                               

for i in scope_objects:                                
    obj = s3_client.get_object(Bucket=bucket, Key=i)
    df = pd.read_parquet(io.BytesIO(obj['Body'].read()))
    dfs.append(df)

In [52]:
semestre1_2024 = dfs[0]
trimestre3_2024 = dfs[1]
trimestre4_2024 = dfs[2]

In [53]:
NB_FER_2024 = pd.concat([
    semestre1_2024,
    trimestre3_2024,
    trimestre4_2024
    ],
    ignore_index= True
)

### ACCESSIBILITE

In [54]:
scope_object_access = [i['Key'] for i in list_objects if i['Key']=="bronze/accessibilite_en_gare.parquet"][0]

In [55]:
obj = s3_client.get_object(Bucket=bucket, Key=scope_object_access)
ACCESSIBILITE= pd.read_parquet(io.BytesIO(obj['Body'].read()))

### REFERENCES

In [56]:
scope_object_references = [i['Key'] for i in list_objects if i['Key']=="bronze/references.parquet"][0]

In [57]:
obj = s3_client.get_object(Bucket=bucket, Key=scope_object_references)
REFERENCES= pd.read_parquet(io.BytesIO(obj['Body'].read()))

## 1. Structure & volume
  - Schéma de chaque fichier (colonnes, types inférés)                                    
  - Nombre de lignes par fichier
---                                             

In [58]:
NB_FER_2024.dtypes

JOUR                object
CODE_STIF_TRNS       int64
CODE_STIF_RES       object
CODE_STIF_ARRET     object
LIBELLE_ARRET       object
ID_ZDC             float64
CATEGORIE_TITRE     object
NB_VALD             object
dtype: object

In [59]:
ACCESSIBILITE.dtypes

stop_point_id               object
accessibility_level_id       int64
accessibility_level_name    object
commentaire                 object
stop_name                   object
stop_point_geopoint         object
dtype: object

In [60]:
REFERENCES.dtypes

zdaid              object
zdaversion         object
zdacreated         object
zdachanged         object
zdaname            object
zdaxepsg2154        int64
zdayepsg2154        int64
zdcid              object
zdapostalregion    object
zdatown            object
zdatype            object
dtype: object

- `NB_VALD` est de type `object` alors qu'il représente un nombre de validations -> à convertir en numérique en Silver(attention aux valeurs non convertibles)
- `ID_ZDC` est en `float64` dans NB_FER_2024 et `zdcid` est en `object` dans REFERENCES —> les types doivent être harmonisés avant la jointure
- La jointure se fait en deux étapes via la table pivot REFERENCES : `NB_FER_2024.ID_ZDC → REFERENCES.zdcid → REFERENCES zdaid → ACCESSIBILITE.stop_point_id`
- NB_FER_2024 est la concaténation de 3 fichiers (S1, T3, T4) —> vérifier l'absence de doublons après concat

In [61]:
NB_FER_2024.shape

(1797296, 8)

In [62]:
ACCESSIBILITE.shape

(459, 6)

In [63]:
REFERENCES.shape

(18045, 11)

- 1,8M de lignes pour NB_FER_2024 — c'est du volume conséquent, confirme l'intérêt d'Athena pour la Silver
- 459 lignes pour ACCESSIBILITE — c'est peu, ce sont les points d'arrêt (ZdA), pas les  stations (ZdC)                                                                          
- 18 045 lignes pour REFERENCES — la table pivot ZdA ↔ ZdC

In [66]:
NB_FER_2024.isnull().sum()

JOUR                  0
CODE_STIF_TRNS        0
CODE_STIF_RES       905
CODE_STIF_ARRET     905
LIBELLE_ARRET         0
ID_ZDC             2236
CATEGORIE_TITRE       0
NB_VALD               0
dtype: int64

In [67]:
ACCESSIBILITE.isnull().sum()

stop_point_id                 0
accessibility_level_id        0
accessibility_level_name      0
commentaire                 442
stop_name                     0
stop_point_geopoint           0
dtype: int64

In [68]:
REFERENCES.isnull().sum()

zdaid              0
zdaversion         0
zdacreated         0
zdachanged         0
zdaname            0
zdaxepsg2154       0
zdayepsg2154       0
zdcid              0
zdapostalregion    0
zdatown            0
zdatype            1
dtype: int64